# Exp 30: Gray non-VIP

Card: [exp30_gray_non_vip.md](../../docs/neuroslope/experiments/exp30_gray_non_vip.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from training.config import ROOT
from training.config import VIP
from training.cv import CVResult
from analysis.gradcam import GradCamFigures
from analysis.plots import pr_curves, save_figures, use_style
from analysis.thresholds import best_f1
from training.config import CLASSES

N_SAMPLE = 16  # exp19 sample
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
GRAY = CONFIG_DIR / "exp30_gray_non_vip" / "resnet18.yaml"
EXP20 = CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp30_gray_non_vip"

# dataviz reference palette, light mode
SERIES = {"exp20": "#2a78d6", "gray": "#1baf7a"}
use_style()

figures = {}

## Training


In [ ]:
results = {"exp20": CVResult.load(EXP20), "gray": CVResult.load(GRAY)}

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Confusion counts, VIP

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
rows = []
for name, r in results.items():
    thresholds = best_f1(r.labels, r.probs, CLASSES)["threshold"]
    for cls in VIP:
        y, pred = r.oof[cls].eq(1), r.oof[f"p_{cls}"] >= thresholds[cls]
        rows.append({
            "model": name, "class": cls,
            "TP": int((y & pred).sum()), "FP": int((~y & pred).sum()),
            "FN": int((y & ~pred).sum()),
        })
confusion = pd.DataFrame(rows).set_index(["class", "model"]).sort_index(level=0, sort_remaining=False)
confusion

## Grad-CAM Cat, Dog: exp20 vs gray


In [ ]:
cams = {"exp20": GradCamFigures(EXP20), "gray": GradCamFigures(GRAY)}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = exp16.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = cams["gray"].positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)